In [1]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, os

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / ".git").exists():
    ROOT = ROOT.parent
assert (ROOT / ".git").exists(), "Repository root not found"

EXPERIMENT_ID = "architecture_v3_free_stable_identity_bridge_qualification_v1"
DESIGN_SIGNATURE = "architecture-v3-free-stable-identity-bridge-qualification-v1:fixed-12-cases:sec-cik+openfigi+finra+nasdaq:effective-alias+event-reason+terminal-value:no-model:no-holdout:no-bulk-before-pass"
QUALIFICATION_DATES = ["2021-08-23", "2024-01-02", "2026-05-28"]
CONSUMED_HOLDOUT = ["2026-05-29", "2026-08-24"]
CASES = [
    {"case":"continuous", "symbols":["AAPL"], "cik":"0000320193"},
    {"case":"continuous", "symbols":["MSFT"], "cik":"0000789019"},
    {"case":"new_listing", "symbols":["ABNB"], "cik":"0001559720"},
    {"case":"new_listing", "symbols":["ARM"], "cik":"0001973239"},
    {"case":"new_listing", "symbols":["CAVA"], "cik":"0001639438"},
    {"case":"ticker_change_reuse", "symbols":["FB","META"], "cik":"0001326801"},
    {"case":"ticker_change", "symbols":["SNE","SONY"], "cik":"0000313838"},
    {"case":"merger_exit", "symbols":["ATVI"], "cik":"0000718877"},
    {"case":"acquisition_exit", "symbols":["TWTR"], "cik":"0001418091"},
    {"case":"bankruptcy_exit", "symbols":["BBBY","BBBYQ"], "cik":"0000886158"},
    {"case":"bankruptcy_exit", "symbols":["WE","WEWKQ"], "cik":"0001813756"},
    {"case":"bankruptcy_exit", "symbols":["REV","REVRQ"], "cik":"0000887921"},
]

spec = {
    "schema_version": 1,
    "experiment_id": EXPERIMENT_ID,
    "design_signature": DESIGN_SIGNATURE,
    "status": "preregistered_approved_after_dependency",
    "objective": "Determine whether free SEC, OpenFIGI, FINRA and Nasdaq evidence can create stable security identities, dated ticker mappings, classified exits and defensible terminal values for a fixed edge-case panel.",
    "qualification_dates": QUALIFICATION_DATES,
    "fixed_cases": CASES,
    "sources": {
        "sec": "submissions JSON and current ticker/exchange file; no authentication",
        "openfigi": "unauthenticated mapping API at public rate limits",
        "finra": "public OTC Daily List API, bounded symbol queries",
        "nasdaq": "current nasdaqlisted and otherlisted symbol directories",
        "alpha_vantage": "no new requests; prior snapshots are discovery evidence only"
    },
    "request_budget": {"sec_max": 13, "openfigi_max": 5, "finra_max": 12, "nasdaq_max": 2},
    "mandatory_gates": [
        "unique stable security-class identity",
        "historical ticker aliases with effective dates",
        "listing and exit dates with event classification",
        "documented terminal value or preregistered conservative treatment",
        "no unresolved ticker reuse or exact-symbol collision",
        "source evidence sufficient for independent rerun"
    ],
    "terminal_value_policy": {
        "documented_cash_or_stock_merger": "use filed consideration terms only",
        "documented_cancelled_bankruptcy_equity": "zero recovery from the last eligible close",
        "unresolved_exit": "exclude from reconstruction and report; never silently drop or assume survival"
    },
    "stop_rule": "Stop before any bulk or 756-date reconstruction if any mandatory gate fails.",
    "governance": {
        "paper_only": True,
        "model_fitting_allowed": False,
        "price_rows_allowed": False,
        "trading_allowed": False,
        "bulk_reconstruction_allowed": False,
        "consumed_holdout_reuse_allowed": False,
        "consumed_holdout_dates_prohibited": CONSUMED_HOLDOUT,
        "credentials_required": False
    }
}

spec_path = ROOT / "research_context" / "architecture_v3_free_stable_identity_bridge_qualification_v1_20260919.json"
candidate_path = ROOT / "research_context" / "context_gate_candidate_update_free_identity_bridge_v1_20260919.json"
gate_path = ROOT / "research_context" / "context_gate.json"

gate = json.loads(gate_path.read_text())
all_rows = gate.get("completed_experiments", []) + gate.get("next_experiments", [])
assert not any(r.get("experiment_id") == EXPERIMENT_ID for r in all_rows), "Duplicate experiment id"
assert not any(r.get("design_signature") == DESIGN_SIGNATURE for r in all_rows), "Duplicate design signature"
entry = {
    "experiment_id": EXPERIMENT_ID,
    "design_signature": DESIGN_SIGNATURE,
    "specification": str(spec_path.relative_to(ROOT)),
    "status": "approved_after_dependency",
    "model_fitting_allowed": False,
    "bulk_reconstruction_allowed": False,
    "promotion_allowed": False,
    "trading_allowed": False,
    "consumed_holdout_reuse_allowed": False
}
gate.setdefault("next_experiments", []).append(entry)
spec_path.write_text(json.dumps(spec, indent=2) + "\n")
candidate_path.write_text(json.dumps({"action":"append_next_experiment","entry":entry}, indent=2) + "\n")
gate_path.write_text(json.dumps(gate, indent=2) + "\n")

module_spec = importlib.util.spec_from_file_location("context_gate", ROOT / "scripts" / "context_gate.py")
context_gate = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(context_gate)
fingerprint = context_gate.assert_experiment_allowed(context_gate.load_gate(gate_path), EXPERIMENT_ID, DESIGN_SIGNATURE, spec)
print(json.dumps({"status":"preregistered", "fingerprint":fingerprint, "cases":len(CASES), "network_requests":0, "holdout_rows_read":0}, indent=2))


{
  "status": "preregistered",
  "fingerprint": "e220bdc1c27004dbce422e26e1bea4543c066f15d754f7e70361f89195a1d218",
  "cases": 12,
  "network_requests": 0,
  "holdout_rows_read": 0
}


In [2]:
from getpass import getpass
if not os.environ.get("SEC_USER_AGENT", "").strip():
    os.environ["SEC_USER_AGENT"] = getpass("SEC user-agent with contact email (hidden): ").strip()
assert "@" in os.environ["SEC_USER_AGENT"]
print("SEC contact configured for this kernel only; value was not displayed or written to disk.")


SEC user-agent with contact email (hidden):  ········


SEC contact configured for this kernel only; value was not displayed or written to disk.


In [3]:
import hashlib, time, urllib.error, urllib.parse, urllib.request

context_gate.assert_experiment_allowed(context_gate.load_gate(gate_path), EXPERIMENT_ID, DESIGN_SIGNATURE, spec)
assert all(not (CONSUMED_HOLDOUT[0] <= d <= CONSUMED_HOLDOUT[1]) for d in QUALIFICATION_DATES)

out_dir = ROOT / "warehouse" / "lineage" / "architecture_v3_free_stable_identity_bridge_qualification_v1_20260919"
raw_dir = out_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
manifest = {
    "experiment_id": EXPERIMENT_ID,
    "design_signature": DESIGN_SIGNATURE,
    "design_fingerprint": fingerprint,
    "qualification_dates": QUALIFICATION_DATES,
    "requests": [],
    "files": [],
    "consumed_holdout_rows_read": 0,
    "models_fit": 0,
    "price_rows_read": 0,
    "trades_or_orders": 0,
    "credentials_stored": False,
}

def request_bytes(url, *, headers=None, body=None, method=None, source, label):
    req = urllib.request.Request(url, data=body, headers=headers or {}, method=method)
    try:
        with urllib.request.urlopen(req, timeout=90) as response:
            payload = response.read()
            status = int(response.status)
        manifest["requests"].append({"source":source,"label":label,"status":status,"bytes":len(payload)})
        return payload
    except Exception as exc:
        manifest["requests"].append({"source":source,"label":label,"status":"error","error_type":type(exc).__name__})
        return None

def save_raw(filename, payload, source, label):
    if payload is None:
        return None
    path = raw_dir / filename
    path.write_bytes(payload)
    manifest["files"].append({"path":str(path.relative_to(ROOT)),"source":source,"label":label,"bytes":len(payload),"sha256":hashlib.sha256(payload).hexdigest()})
    return path

sec_headers = {"User-Agent": os.environ["SEC_USER_AGENT"], "Accept-Encoding":"gzip, deflate"}
sec_tickers = request_bytes("https://www.sec.gov/files/company_tickers_exchange.json", headers=sec_headers, source="sec", label="current_ticker_exchange")
save_raw("sec_company_tickers_exchange.json", sec_tickers, "sec", "current_ticker_exchange")
for row in CASES:
    cik = row["cik"]
    payload = request_bytes(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=sec_headers, source="sec", label=f"submissions_{cik}")
    save_raw(f"sec_submissions_CIK{cik}.json", payload, "sec", f"submissions_{cik}")
    time.sleep(0.15)

symbols = sorted({s for row in CASES for s in row["symbols"]})
figi_url = "https://api.openfigi.com/v3/mapping"
figi_headers = {"Content-Type":"application/json", "User-Agent":"stockprediction2025-public-research/1.0"}
for batch_no, start in enumerate(range(0, len(symbols), 5), 1):
    batch = symbols[start:start+5]
    jobs = [{"idType":"TICKER","idValue":s,"exchCode":"US","marketSecDes":"Equity"} for s in batch]
    payload = request_bytes(figi_url, headers=figi_headers, body=json.dumps(jobs).encode(), method="POST", source="openfigi", label=f"batch_{batch_no}_{'_'.join(batch)}")
    save_raw(f"openfigi_mapping_batch_{batch_no}.json", payload, "openfigi", f"batch_{batch_no}")
    if start + 5 < len(symbols):
        time.sleep(3)

finra_url = "https://api.finra.org/data/group/otcMarket/name/otcDailyList"
finra_symbols = ["FB","SNE","ATVI","TWTR","BBBY","BBBYQ","WE","WEWKQ","REV","REVRQ"]
finra_fields = ["calendarDay","dailyListEventCode","dailyListReasonDescription","oldSymbolCode","newSymbolCode","oldSecurityDescription","newSecurityDescription","securityDeleteFlag","securityAddFlag","changeSymbolFlag","bankruptcyFlag","cashAmountText","exDate","paymentDate","commentText"]
for sym in finra_symbols:
    body = {"limit":1000,"fields":finra_fields,"compareFilters":[{"compareType":"EQUAL","fieldName":"oldSymbolCode","fieldValue":sym}]}
    payload = request_bytes(finra_url, headers={"Content-Type":"application/json","User-Agent":"stockprediction2025-public-research/1.0"}, body=json.dumps(body).encode(), method="POST", source="finra", label=f"old_symbol_{sym}")
    save_raw(f"finra_otc_daily_list_old_symbol_{sym}.json", payload, "finra", f"old_symbol_{sym}")

nasdaq_urls = {
    "nasdaqlisted":"https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt",
    "otherlisted":"https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt",
}
for label, url in nasdaq_urls.items():
    payload = request_bytes(url, headers={"User-Agent":"stockprediction2025-public-research/1.0"}, source="nasdaq", label=label)
    save_raw(f"nasdaq_{label}.txt", payload, "nasdaq", label)

manifest["collected_at_utc"] = datetime.now(timezone.utc).isoformat()
(out_dir / "collection_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps({
    "status":"bounded_collection_complete",
    "request_count":len(manifest["requests"]),
    "successful_requests":sum(r["status"] == 200 for r in manifest["requests"]),
    "errors":[r for r in manifest["requests"] if r["status"] != 200],
    "files":len(manifest["files"]),
    "holdout_rows_read":0,
    "models_fit":0,
    "price_rows_read":0,
}, indent=2))


{
  "status": "bounded_collection_complete",
  "request_count": 29,
  "successful_requests": 22,
  "errors": [
    {
      "source": "finra",
      "label": "old_symbol_FB",
      "status": 204,
      "bytes": 0
    },
    {
      "source": "finra",
      "label": "old_symbol_SNE",
      "status": 204,
      "bytes": 0
    },
    {
      "source": "finra",
      "label": "old_symbol_ATVI",
      "status": 204,
      "bytes": 0
    },
    {
      "source": "finra",
      "label": "old_symbol_TWTR",
      "status": 204,
      "bytes": 0
    },
    {
      "source": "finra",
      "label": "old_symbol_BBBY",
      "status": 204,
      "bytes": 0
    },
    {
      "source": "finra",
      "label": "old_symbol_WE",
      "status": 204,
      "bytes": 0
    },
    {
      "source": "finra",
      "label": "old_symbol_REV",
      "status": 204,
      "bytes": 0
    }
  ],
  "files": 29,
  "holdout_rows_read": 0,
  "models_fit": 0,
  "price_rows_read": 0
}


In [4]:
import gzip, io, csv, pandas as pd

def decoded_json(path):
    payload = path.read_bytes()
    if not payload:
        return None
    if payload[:2] == b"\x1f\x8b":
        payload = gzip.decompress(payload)
    text = payload.decode("utf-8-sig")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return list(csv.DictReader(io.StringIO(text)))

sec_cases = []
for row in CASES:
    p = raw_dir / f"sec_submissions_CIK{row['cik']}.json"
    doc = decoded_json(p)
    recent = doc.get("filings", {}).get("recent", {}) if doc else {}
    forms = recent.get("form", [])
    dates = recent.get("filingDate", [])
    accessions = recent.get("accessionNumber", [])
    relevant = [
        {"form":f,"filingDate":d,"accessionNumber":a}
        for f,d,a in zip(forms,dates,accessions)
        if f in {"25","25-NSE","15-12G","8-K","8-K/A","6-K","20-F","10-K","10-K/A"}
    ][:25]
    sec_cases.append({
        "case":row["case"], "requested_cik":row["cik"],
        "entityName":doc.get("name") if doc else None,
        "tickers":doc.get("tickers",[]) if doc else [],
        "exchanges":doc.get("exchanges",[]) if doc else [],
        "formerNames":doc.get("formerNames",[]) if doc else [],
        "relevant_filings_sample":relevant,
        "has_form25_or_15":any(x["form"] in {"25","25-NSE","15-12G"} for x in relevant),
    })

figi_rows = []
for batch_no, start in enumerate(range(0, len(symbols), 5), 1):
    batch = symbols[start:start+5]
    doc = decoded_json(raw_dir / f"openfigi_mapping_batch_{batch_no}.json") or []
    for symbol, response in zip(batch, doc):
        matches = response.get("data", []) if isinstance(response, dict) else []
        figi_rows.append({
            "symbol":symbol,
            "match_count":len(matches),
            "matches":[{k:m.get(k) for k in ["figi","compositeFIGI","shareClassFIGI","ticker","name","exchCode","securityType2"]} for m in matches[:10]],
            "error":response.get("error") if isinstance(response, dict) else "invalid_response",
        })

finra_rows = []
for sym in finra_symbols:
    doc = decoded_json(raw_dir / f"finra_otc_daily_list_old_symbol_{sym}.json")
    rows = doc if isinstance(doc, list) else []
    finra_rows.append({"symbol":sym,"row_count":len(rows),"rows":rows[:20]})

nasdaq_current = {}
for label in nasdaq_urls:
    text = (raw_dir / f"nasdaq_{label}.txt").read_text(errors="replace")
    lines = [ln for ln in text.splitlines() if "|" in ln and not ln.startswith("File Creation Time")]
    header = lines[0].split("|")
    records = [dict(zip(header, ln.split("|"))) for ln in lines[1:]]
    symbol_field = "Symbol" if label == "nasdaqlisted" else "ACT Symbol"
    nasdaq_current[label] = {r.get(symbol_field):r for r in records if r.get(symbol_field)}

figi_summary = {r["symbol"]:{"match_count":r["match_count"],"figis":sorted({m.get("figi") for m in r["matches"] if m.get("figi")}),"names":sorted({m.get("name") for m in r["matches"] if m.get("name")})} for r in figi_rows}
sec_summary = [{"case":r["case"],"cik":r["requested_cik"],"entity":r["entityName"],"tickers":r["tickers"],"former_name_count":len(r["formerNames"]),"has_form25_or_15":r["has_form25_or_15"]} for r in sec_cases]
finra_summary = {r["symbol"]:r["row_count"] for r in finra_rows}
print(json.dumps({
    "sec_cases":sec_summary,
    "openfigi":figi_summary,
    "finra_rows_by_old_symbol":finra_summary,
    "current_nasdaq_presence":{s:{k:(s in v) for k,v in nasdaq_current.items()} for s in symbols},
    "holdout_rows_read":0,
}, indent=2))


In [6]:
import csv

def decoded_records(path):
    payload = path.read_bytes()
    if not payload:
        return []
    if payload[:2] == b"\x1f\x8b":
        payload = gzip.decompress(payload)
    text = payload.decode("utf-8-sig")
    try:
        doc = json.loads(text)
        return doc if isinstance(doc, list) else []
    except json.JSONDecodeError:
        return list(csv.DictReader(io.StringIO(text)))

finra_rows = []
for sym in finra_symbols:
    rows = decoded_records(raw_dir / f"finra_otc_daily_list_old_symbol_{sym}.json")
    finra_rows.append({"symbol":sym,"row_count":len(rows),"rows":rows[:20]})

nasdaq_current = {}
for label in nasdaq_urls:
    text = (raw_dir / f"nasdaq_{label}.txt").read_text(errors="replace")
    lines = [ln for ln in text.splitlines() if "|" in ln and not ln.startswith("File Creation Time")]
    header = lines[0].split("|")
    records = [dict(zip(header, ln.split("|"))) for ln in lines[1:]]
    symbol_field = "Symbol" if label == "nasdaqlisted" else "ACT Symbol"
    nasdaq_current[label] = {r.get(symbol_field):r for r in records if r.get(symbol_field)}

figi_summary = {r["symbol"]:{"match_count":r["match_count"],"figis":sorted({m.get("figi") for m in r["matches"] if m.get("figi")}),"names":sorted({m.get("name") for m in r["matches"] if m.get("name")})} for r in figi_rows}
sec_summary = [{"case":r["case"],"cik":r["requested_cik"],"entity":r["entityName"],"tickers":r["tickers"],"former_name_count":len(r["formerNames"]),"has_form25_or_15":r["has_form25_or_15"]} for r in sec_cases]
finra_summary = {r["symbol"]:r["row_count"] for r in finra_rows}
print(json.dumps({
    "sec_cases":sec_summary,
    "openfigi":figi_summary,
    "finra_rows_by_old_symbol":finra_summary,
    "finra_nonempty":[r for r in finra_rows if r["row_count"]],
    "current_nasdaq_presence":{s:{k:(s in v) for k,v in nasdaq_current.items()} for s in symbols},
    "holdout_rows_read":0,
}, indent=2))


{
  "sec_cases": [
    {
      "case": "continuous",
      "cik": "0000320193",
      "entity": "Apple Inc.",
      "tickers": [
        "AAPL"
      ],
      "former_name_count": 3,
      "has_form25_or_15": true
    },
    {
      "case": "continuous",
      "cik": "0000789019",
      "entity": "MICROSOFT CORP",
      "tickers": [
        "MSFT"
      ],
      "former_name_count": 0,
      "has_form25_or_15": false
    },
    {
      "case": "new_listing",
      "cik": "0001559720",
      "entity": "Airbnb, Inc.",
      "tickers": [
        "ABNB"
      ],
      "former_name_count": 1,
      "has_form25_or_15": false
    },
    {
      "case": "new_listing",
      "cik": "0001973239",
      "entity": "ARM HOLDINGS PLC /UK",
      "tickers": [
        "ARM"
      ],
      "former_name_count": 1,
      "has_form25_or_15": false
    },
    {
      "case": "new_listing",
      "cik": "0001639438",
      "entity": "CAVA GROUP, INC.",
      "tickers": [
        "CAVA"
      ],
      "forme

In [11]:
completed_at = datetime.now(timezone.utc).isoformat()
result_rel = "research_context/architecture_v3_free_stable_identity_bridge_qualification_result_v1_20260919.json"
result_md_rel = "research_context/architecture_v3_free_stable_identity_bridge_qualification_result_v1_20260919.md"
case_results = [
 {"case":"AAPL continuous","status":"partial","finding":"SEC CIK and OpenFIGI identify the current security, but do not prove point-in-time membership on all qualification dates."},
 {"case":"MSFT continuous","status":"partial","finding":"SEC CIK and OpenFIGI identify the current security, but do not prove point-in-time membership on all qualification dates."},
 {"case":"ABNB new listing","status":"fail","finding":"Current identifiers found; no authoritative listing-effective date returned."},
 {"case":"ARM new listing","status":"fail","finding":"Current identifiers found; no authoritative listing-effective date returned."},
 {"case":"CAVA new listing","status":"fail","finding":"Current identifiers found; no authoritative listing-effective date returned."},
 {"case":"FB to META / ticker reuse","status":"fail","finding":"SEC shows only current META. Current OpenFIGI maps FB to ProShares S&P Dynamic Buffer (BBG01VRMNFB1), not Meta (BBG000MM2P62); no effective-dated alias bridge exists."},
 {"case":"SNE to SONY","status":"fail","finding":"SEC shows current SONY/SNEJF and OpenFIGI resolves SONY, but SNE is absent and no effective-dated alias bridge exists."},
 {"case":"ATVI merger exit","status":"fail","finding":"SEC current tickers are empty and Form 25/15 evidence exists, but bounded metadata supplies no merger consideration or terminal return."},
 {"case":"TWTR acquisition exit","status":"fail","finding":"SEC current tickers are empty and Form 25/15 evidence exists, but bounded metadata supplies no acquisition consideration or terminal return."},
 {"case":"BBBY/BBBYQ bankruptcy","status":"partial","finding":"FINRA records OTC entry and 2023-10-02 shares-cancelled deletion, but cash is blank and the original-to-Q bridge is not stable/effective-dated."},
 {"case":"WE/WEWKQ bankruptcy","status":"partial","finding":"FINRA records OTC entry and 2024-06-11 shares-cancelled deletion, but cash is blank and the original-to-Q bridge is not stable/effective-dated."},
 {"case":"REV/REVRQ bankruptcy","status":"partial","finding":"FINRA records OTC entry and 2023-05-02 shares-cancelled deletion, but cash is blank and the original-to-Q bridge is not stable/effective-dated."}
]
gates = {
 "stable_security_class_identity":{"status":"fail","evidence":"SEC CIK is issuer-level; OpenFIGI returned current identifiers only and zero matches for delisted/legacy symbols."},
 "historical_ticker_aliases_with_effective_dates":{"status":"fail","evidence":"SEC and OpenFIGI supplied no effective-dated ticker history; FB currently maps to a different fund."},
 "listing_and_delisting_events":{"status":"partial","evidence":"SEC Form 25/15 and FINRA provide some exits, but new-listing dates and exchange-to-OTC continuity are incomplete."},
 "adjusted_and_unadjusted_prices":{"status":"fail","evidence":"This bridge supplies metadata, not trustworthy adjusted and unadjusted histories."},
 "split_and_dividend_handling":{"status":"fail","evidence":"No endpoint supplied a complete point-in-time corporate-action adjustment chain."},
 "terminal_or_delisting_values":{"status":"fail","evidence":"FINRA cancellation rows have blank cash amounts; ATVI and TWTR consideration was not returned."},
 "ticker_reuse_and_case_collision_resolution":{"status":"fail","evidence":"Current FB resolves to a ProShares fund while historical FB was Meta; case-normalization history is absent."},
 "three_anchor_date_point_in_time_membership":{"status":"fail","evidence":"Nasdaq files are current and the stack cannot reconstruct membership on the three dates."},
 "deterministic_auditability":{"status":"pass","evidence":"All 29 responses were hashed and saved with a manifest; no credentials were persisted."}
}
source_coverage = {
 "SEC":{"requests":13,"http_200":13,"coverage":"12 CIK submissions plus current ticker file; stable issuer CIK and filings, but current ticker fields only."},
 "OpenFIGI":{"requests":4,"http_200":4,"coverage":"17 symbol jobs; 8 current matches and 9 zero matches; no effective dates."},
 "FINRA":{"requests":10,"http_200":3,"http_204":7,"coverage":"Two rows each for BBBYQ, WEWKQ, REVRQ; exchange-listed legacy symbols absent."},
 "Nasdaq":{"requests":2,"http_200":2,"coverage":"Current symbol directories only; not historical membership."}
}
result = {
 "schema_version":"1.0","experiment_id":EXPERIMENT_ID,"design_signature":DESIGN_SIGNATURE,"design_fingerprint":fingerprint,"completed_at_utc":completed_at,
 "decision":"fail_stop_before_full_reconstruction","status":"completed_source_qualification_failed_do_not_repeat","full_reconstruction_authorized":False,"architecture_v3_data_port_authorized":False,
 "qualification_dates":QUALIFICATION_DATES,"consumed_holdout":{"start":"2026-05-29","end":"2026-08-24","rows_read":0,"reuse_allowed":False},
 "safety":{"models_fit":0,"price_rows_read":0,"trades_or_orders":0,"purchases_or_subscriptions":0,"credentials_persisted":False},
 "source_coverage":source_coverage,"gate_results":gates,"case_results":case_results,
 "critical_findings":["The free stack does not yield one effective-dated security master joining issuer, security class, ticker aliases, membership, and exits.","FB demonstrates live ticker reuse: current OpenFIGI identifies an unrelated ProShares fund, so symbol-only joins would corrupt Meta history.","FINRA supplies valuable OTC bankruptcy cancellation events, but not a complete exchange-listed event chain or cash terminal values.","Current Nasdaq directories cannot establish membership on the three historical qualification dates."],
 "allowed_reuse":["Use SEC CIK as an issuer anchor, not as a security-class identifier.","Use OpenFIGI only after an effective-dated issuer/security mapping is independently established.","Use FINRA OTC cancellation rows as event evidence under a prespecified conservative zero-recovery rule when security continuity is independently verified."],
 "next_safe_route":"Wait for WRDS/CRSP or another licensed point-in-time security master; meanwhile run only prospective shadow collection from 2026-09-19 onward. Do not backfill 756 dates from these free sources.",
 "manifest":"warehouse/lineage/architecture_v3_free_stable_identity_bridge_qualification_v1_20260919/collection_manifest.json",
 "official_source_urls":["https://www.sec.gov/search-filings/edgar-application-programming-interfaces","https://www.openfigi.com/api/documentation","https://developer.finra.org/docs","https://www.nasdaqtrader.com/Trader.aspx?id=SymbolDirDefs"]
}
(ROOT / result_rel).write_text(json.dumps(result, indent=2) + "\n")
(out_dir / "qualification_summary.json").write_text(json.dumps(result, indent=2) + "\n")
md = "# Architecture v3 free stable-identity bridge qualification\n\n**Decision:** FAIL - stop before full reconstruction.\n\nThe bounded free-source stack is useful for issuer anchoring and selected exit evidence, but it cannot reconstruct a trustworthy point-in-time US equity security master. No model or price data was read, and the consumed 2026-05-29 through 2026-08-24 holdout remained untouched.\n\n## Mandatory gates\n\n| Gate | Result | Evidence |\n|---|---|---|\n"
md += "\n".join(f"| {k.replace('_',' ')} | {v['status'].upper()} | {v['evidence']} |" for k,v in gates.items())
md += "\n\n## Decisive findings\n\n- SEC CIK is a useful issuer anchor, but ticker fields are current and CIK is not a security-class identifier.\n- OpenFIGI mapped current FB to ProShares S&P Dynamic Buffer, not Meta. Symbol reuse cannot be resolved without effective dates.\n- FINRA found cancellation events for BBBYQ, WEWKQ, and REVRQ, but cash amounts were blank and continuity was incomplete.\n- Nasdaq directories are current snapshots, not historical membership.\n\n## Research decision\n\nDo not run a broad reconstruction or port Architecture v3 from this stack. Wait for WRDS/CRSP or another licensed point-in-time security master. A prospective shadow universe beginning 2026-09-19 is allowed, but it cannot repair the historical 756-date requirement.\n\n## Safety record\n\n- Holdout rows read: 0\n- Models fit: 0\n- Price rows read: 0\n- Trades/orders: 0\n- Purchases/subscriptions: 0\n- Credentials persisted: no\n"
(ROOT / result_md_rel).write_text(md)
idx = next(i for i,x in enumerate(gate_doc["next_experiments"]) if x.get("experiment_id") == EXPERIMENT_ID)
prereg = gate_doc["next_experiments"].pop(idx)
completed_record = {**prereg,"status":"completed_source_qualification_failed_do_not_repeat","completed_on":"2026-09-19","result_path":result_rel,"decision":result["decision"],"full_reconstruction_authorized":False,"consumed_holdout_rows_read":0,"conclusion":"Free SEC/OpenFIGI/FINRA/Nasdaq sources cannot supply effective-dated security identity, full historical membership, and terminal values; stop before bulk reconstruction."}
if not any(x.get("experiment_id") == EXPERIMENT_ID for x in gate_doc["completed_experiments"]): gate_doc["completed_experiments"].append(completed_record)
gate_doc["updated_at"] = completed_at; gate_doc["updated_at_utc"] = completed_at
gate_path.write_text(json.dumps(gate_doc, indent=2) + "\n")
state_doc["architecture_v3_free_stable_identity_bridge_qualification"] = {"status":result["status"],"decision":result["decision"],"result_artifact":result_rel,"critical_findings":result["critical_findings"],"full_reconstruction_authorized":False,"consumed_holdout_rows_read":0}
state_doc["updated_at_utc"] = completed_at
state_path.write_text(json.dumps(state_doc, indent=2) + "\n")
print(json.dumps({"decision":result["decision"],"result":result_rel,"markdown":result_md_rel,"holdout_rows_read":0,"gate_updated":True,"state_updated":True}, indent=2))


{
  "decision": "fail_stop_before_full_reconstruction",
  "result": "research_context/architecture_v3_free_stable_identity_bridge_qualification_result_v1_20260919.json",
  "markdown": "research_context/architecture_v3_free_stable_identity_bridge_qualification_result_v1_20260919.md",
  "holdout_rows_read": 0,
  "gate_updated": true,
  "state_updated": true
}


In [12]:
saved_result = json.loads((ROOT / result_rel).read_text())
saved_gate = json.loads(gate_path.read_text())
saved_state = json.loads(state_path.read_text())
manifest_check = json.loads((out_dir / "collection_manifest.json").read_text())
assert saved_result["decision"] == "fail_stop_before_full_reconstruction"
assert saved_result["consumed_holdout"]["rows_read"] == 0
assert not saved_result["full_reconstruction_authorized"]
assert not any(x.get("experiment_id") == EXPERIMENT_ID for x in saved_gate["next_experiments"])
assert sum(x.get("experiment_id") == EXPERIMENT_ID for x in saved_gate["completed_experiments"]) == 1
assert saved_state["architecture_v3_free_stable_identity_bridge_qualification"]["consumed_holdout_rows_read"] == 0
assert len(manifest_check["requests"]) == 29 and len(manifest_check["files"]) == 29
print(json.dumps({"validation":"pass","decision":saved_result["decision"],"requests":29,"saved_files":29,"holdout_rows_read":0,"models_fit":0,"bulk_reconstruction_started":False}, indent=2))


{
  "validation": "pass",
  "decision": "fail_stop_before_full_reconstruction",
  "requests": 29,
  "saved_files": 29,
  "holdout_rows_read": 0,
  "models_fit": 0,
  "bulk_reconstruction_started": false
}
